# Grouped Query Attention
*Multiple query heads share one key-value head — the memory-efficient middle ground between MHA and MQA.*

---
**Component:** `learning/leetgpu/grouped_query_attention.ipynb`
**Builds on:** `naive_gemm.py`, `softmax_attention_cutedsl.py`


## Abstract

*Imagine an office where every salesperson (query head) has their own dedicated filing cabinet full of customer records (key-value cache). That's Multi-Head Attention. Grouped Query Attention says: why not share filing cabinets? Six salespeople can share one cabinet as long as they take turns looking things up. You go from 32 cabinets to 8, freeing up three-quarters of the office space — while each salesperson still has their own unique way of querying the shared records. At inference time, smaller KV caches mean less data to shuttle from memory, which directly translates to faster token generation.*

---

## What Is GQA?

During autoregressive inference, the model reads stored key and value vectors for every token it has generated so far. The more query heads you have, the more KV vectors you must store — and reading them from HBM is the bottleneck, not the arithmetic.

Standard Multi-Head Attention (MHA) gives every query head its own dedicated key and value head. If you have 32 query heads, you have 32 KV heads — and your KV cache is proportionally large. Multi-Query Attention (MQA) goes to the opposite extreme: one shared KV head for everyone. Quality suffers. Grouped Query Attention (GQA) splits the difference:

```
MHA  (G = H = 32):  Q0→K0   Q1→K1  ... Q31→K31     32 KV heads
GQA  (G = 8):       Q0→K0   Q1→K0   Q2→K0   Q3→K0  ←─ group 0
                    Q4→K1   Q5→K1   Q6→K1   Q7→K1  ←─ group 1
                    ...                              8 KV heads total
MQA  (G = 1):       Q0→K0   Q1→K0  ... Q31→K0       1 KV head
```

The routing rule is just one line: `kv_h = query_h // group_size`. Every query head in the same group reads from the same KV head, reducing the KV cache by a factor of `H/G`.

**Where you'll see it:** LLaMA-2/3, Mistral, DeepSeek — any model designed for long-context generation at scale.

## Level 0 — Pure Python, No Imports

Let's start from the very bottom: nested `for` loops, plain Python lists, no libraries. The goal is to make the `kv_h = h // group_size` routing as visible as possible before we start optimizing.

In [ ]:
import math

# ── tiny fixed inputs ──
T, d_head, H, G = 3, 2, 4, 2   # seq_len=3, head_dim=2, 4 Q-heads, 2 KV-heads
group_size = H // G              # 2 query heads share each KV head

# Q[head][position][dim]
Q = [
    [[0.6, 0.3], [0.1, 0.9], [0.4, 0.5]],   # head 0  → KV-head 0
    [[0.8, 0.2], [0.5, 0.4], [0.3, 0.7]],   # head 1  → KV-head 0
    [[0.2, 0.7], [0.6, 0.1], [0.9, 0.3]],   # head 2  → KV-head 1
    [[0.5, 0.6], [0.3, 0.8], [0.7, 0.2]],   # head 3  → KV-head 1
]
# K[kv_head][position][dim],  V[kv_head][position][dim]
K = [
    [[0.7, 0.4], [0.2, 0.8], [0.5, 0.3]],   # kv_head 0
    [[0.3, 0.6], [0.8, 0.1], [0.4, 0.7]],   # kv_head 1
]
V = [
    [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]],
    [[0.8, 0.2], [0.3, 0.7], [0.6, 0.4]],
]

output = [[[0.0] * d_head for _ in range(T)] for _ in range(H)]

for h in range(H):
    kv_h = h // group_size          # ← the only GQA-specific line
    for t in range(T):
        # dot products
        scores = []
        for s in range(T):
            score = 0.0
            for k in range(d_head):
                score += Q[h][t][k] * K[kv_h][s][k]
            scores.append(score / math.sqrt(d_head))
        # stable softmax
        m = max(scores)
        exps = [math.exp(sc - m) for sc in scores]
        Z = sum(exps)
        weights = [e / Z for e in exps]
        # weighted value sum
        for k in range(d_head):
            output[h][t][k] = sum(weights[s] * V[kv_h][s][k] for s in range(T))

print("GQA output[head][token][dim]:")
for h in range(H):
    for t in range(T):
        print(f"  head={h}(KV={h//group_size}) token={t}: {[round(v, 4) for v in output[h][t]]}")


## Verbose Version — Tracing the Sharing

The code above computed the right answer, but you had to squint to see the sharing happen. Let's slow it down and print exactly which KV head each query head reads from at each step. This makes the group structure impossible to miss.

In [ ]:
print("=== GQA attention trace ===")
for h in range(H):
    kv_h = h // group_size
    print(f"\n[Q-head {h}] reads K/V from kv_head={kv_h}  (group_size={group_size})")
    for t in range(T):
        # scores
        scores = []
        for s in range(T):
            raw = sum(Q[h][t][k] * K[kv_h][s][k] for k in range(d_head))
            score = raw / math.sqrt(d_head)
            scores.append(score)
            print(f"  Q[{h}][t={t}] · K[{kv_h}][s={s}] / sqrt({d_head}) = {score:.4f}")
        # softmax
        m = max(scores)
        exps = [math.exp(sc - m) for sc in scores]
        Z = sum(exps)
        weights = [e / Z for e in exps]
        print(f"  attn_weights t={t}: {[round(w, 4) for w in weights]}")


## The Formula

$$\boxed{\text{GQA}(Q, K, V)_h = \text{softmax}\!\left(\frac{Q_h\, K_{\lfloor h/g \rfloor}^\top}{\sqrt{d_k}}\right) V_{\lfloor h/g \rfloor}}$$

The only difference from standard attention is the subscript on $K$ and $V$. Instead of indexing by $h$ (the query head), you index by $\lfloor h/g \rfloor$ — the group that head $h$ belongs to.

| Symbol | Meaning |
|--------|---------|
| $H$ | Total query heads |
| $G$ | KV heads ($G \leq H$, $G$ divides $H$) |
| $g = H/G$ | Queries per KV group (the group size) |
| $Q_h \in \mathbb{R}^{T \times d_k}$ | Queries for head $h$ |
| $K_{\lfloor h/g \rfloor}$ | Keys shared by the $g$ query heads in the same group |
| $V_{\lfloor h/g \rfloor}$ | Values shared by the same group |

> **Say it out loud:** "GQA output for query head $h$ is standard softmax attention, except the key and value matrices are looked up by $\lfloor h/g \rfloor$ instead of $h$ — that floor operation is literally the only change."

**Practical savings.** At $H = 32$, $G = 8$, the KV cache shrinks by $32/8 = 4\times$. For a 70B model running at batch size 1, that can mean the difference between fitting on two GPUs versus needing eight.

## CuTe Layout Python Emulation

Before going to actual GPU code, it's useful to express the same computation using explicit `Layout` and `Tensor` objects — a Python proxy for how CuTe thinks about memory. The head dimension maps to a row offset, the group mapping becomes an integer division in the stride calculation. This step bridges "what the math says" and "what the hardware sees."

In [ ]:
class Layout:
    def __init__(self, shape, stride=None):
        self.shape = tuple(shape)
        if stride is None:
            s, running = [], 1
            for d in reversed(shape):
                s.insert(0, running); running *= d
            self.stride = tuple(s)
        else:
            self.stride = tuple(stride)
    def __call__(self, *coord):
        return sum(c * s for c, s in zip(coord, self.stride))
    def __repr__(self): return f"Layout(shape={self.shape}, stride={self.stride})"

def softmax_vec(v):
    m = max(v); exps = [math.exp(x - m) for x in v]; Z = sum(exps)
    return [e / Z for e in exps]

def gqa_cute_layout(Q_flat, K_flat, V_flat, H, G, T, d):
    group_size = H // G
    layout_Q  = Layout([H, T, d])   # row-major over (head, token, dim)
    layout_KV = Layout([G, T, d])   # KV heads
    out = [0.0] * (H * T * d)
    layout_out = Layout([H, T, d])

    for h in range(H):
        kv_h = h // group_size
        for t in range(T):
            scores = []
            for s in range(T):
                raw = sum(
                    Q_flat[layout_Q(h, t, k)] * K_flat[layout_KV(kv_h, s, k)]
                    for k in range(d)
                )
                scores.append(raw / math.sqrt(d))
            weights = softmax_vec(scores)
            for dk in range(d):
                out[layout_out(h, t, dk)] = sum(
                    weights[s] * V_flat[layout_KV(kv_h, s, dk)] for s in range(T)
                )
    return out

# Flatten our Q/K/V lists into 1-D sequences
Q_flat = [Q[h][t][k] for h in range(H) for t in range(T) for k in range(d_head)]
K_flat = [K[g][t][k] for g in range(G) for t in range(T) for k in range(d_head)]
V_flat = [V[g][t][k] for g in range(G) for t in range(T) for k in range(d_head)]

out_flat = gqa_cute_layout(Q_flat, K_flat, V_flat, H, G, T, d_head)
print("CuTe Layout GQA output (head=0, all tokens):")
lo = Layout([H, T, d_head])
for t in range(T):
    vals = [round(out_flat[lo(0, t, k)], 4) for k in range(d_head)]
    print(f"  token {t}: {vals}")

# cross-check against naive output
tol = 1e-9
for h in range(H):
    for t in range(T):
        for k in range(d_head):
            assert abs(out_flat[lo(h, t, k)] - output[h][t][k]) < tol
print("✓ matches naive output")


## CuTeDSL Implementation

In the CuTeDSL kernel, each GPU thread is responsible for exactly one output element `out[h, m, d_out]`. The thread computes the full two-pass online softmax for its row, accessing `mK[kv_h, s, k]` and `mV[kv_h, s, d_out]` where `kv_h = h // group_size`. That one substitution is all that distinguishes this from the `softmax_attention_cutedsl.py` baseline.

```python
@cute.kernel
def gqa_kernel(mQ, mK, mV, mOut, scale, group_size):
    ...
    kv_h = h // group_size   # ← the only GQA-specific line
    ...
    for s in cutlass.range(N, unroll=1):
        score = score + mQ[h,m,k] * mK[kv_h,s,k]   # shared KV head
    ...
    acc = acc + p * mV[kv_h, s, d_out]              # shared values
```

> **Performance note.** Having fewer KV heads means fewer distinct cache lines to load from HBM. When $H/G = 4$, four threads share the same KV data — and with good scheduling, the L2 cache can serve three of those four requests for free after the first fetch.

## Optimization Ladder

| Version | KV-head index | Threads | Notes |
|---------|--------------|---------|-------|
| Pure Python | `h // group_size` | 1 (serial) | Readable reference |
| CuTe Layout | same | 1 (serial Python) | Verifies stride/offset math |
| `gqa_kernel` (CuTeDSL) | `kv_h = h // group_size` | 1 per output element | Direct port of `softmax_attention_cutedsl.py` |

**KV cache at inference time** — what you're actually saving:

| Config | KV heads | Cache per layer per token | vs MHA |
|--------|---------|--------------------------|--------|
| MHA $H=32$ | 32 | $2 \times 32 \times d_k$ floats | — |
| GQA $H=32, G=8$ | 8 | $2 \times 8 \times d_k$ floats | **4× smaller** |
| MQA $H=32, G=1$ | 1 | $2 \times 1 \times d_k$ floats | **32× smaller** |

A 4× KV reduction directly translates to 4× more sequences you can serve simultaneously on the same GPU, or 4× longer context at the same memory budget.

## KV Cache Bandwidth at Decode: Where the Time Actually Goes

When generating one token, attention must read the entire KV cache — every key and value
stored from all previous tokens — for every head in every layer.

**Qwen3-8B KV cache size** (`n_kv_heads=8`, `head_dim=128`, BF16):
```
Per layer: 2 (K+V) × T_ctx tokens × 8 heads × 128 dims × 2 bytes

  T_ctx =  1,000:   4 MB/layer  × 36 layers =  144 MB  →  0.38 ms per token
  T_ctx =  4,096:  16 MB/layer  × 36 layers =  576 MB  →  1.52 ms per token
  T_ctx = 32,768: 128 MB/layer  × 36 layers =  4.6 GB  → 12.1  ms per token
```

No kernel optimization can reduce this. The bytes exist, and they must be read.

**GQA's direct impact:**
Qwen3-8B has 8 KV heads, not 32. That's a 4× reduction vs standard MHA:
```
MHA  (32 KV heads) at T_ctx=4096: 2,304 MB → 6.1 ms per token
GQA  ( 8 KV heads) at T_ctx=4096:   576 MB → 1.52 ms per token
```

**FP8 KV cache halves it again:**
Store K and V in FP8 (1 byte) instead of BF16 (2 bytes):
```
FP8 KV at T_ctx=4096: 288 MB → 0.76 ms per token
```

FP8 KV quality holds up well in practice. Attention is a weighted average over many positions,
so small errors in individual K or V values tend to cancel out rather than compound.

## SM Utilization at Decode: GQA Means Fewer CTAs

At decode, the attention kernel generates one output token.
Each query head needs its own CTA — so with H=32 query heads, we launch 32 CTAs.
The RTX 4080 has 76 SMs, so 44 of them sit idle. We're at 42% utilization.

**It's actually worse.** Inside each CTA, the KV loop is sequential:
load tile 0, compute tile 0, load tile 1, compute tile 1, ...
At T_ctx=4096 with Bc=64, that's 64 iterations per CTA.
No other CTA can steal those 64 slots — the work is serialized inside each head.

**Three ways to improve:**

**1. Flash-Decoding (Split-KV):**
Split the context into chunks and assign each chunk to a separate CTA.
Then reduce the partial results from each chunk at the end.

```
4 chunks of T_ctx/4 = 1024 tokens each:
  CTAs: 32 heads × 4 chunks = 128
  SM utilization: 128 / 76 ≈ 1.7×  (much better)
```

**2. Batch multiple requests:**
Serve 8 users at once. Each user needs 32 CTAs → 256 CTAs total → GPU fully utilized.
Continuous batching is the most practical way to saturate the RTX 4080 at decode.

**3. Persistent kernels:**
Keep CTAs alive across many decode steps instead of relaunching for each token.
This amortizes the 5–20 µs kernel launch overhead across hundreds of tokens.

## Where GQA Lives in the Qwen3-8B Decode Step

GQA is the attention computation at step ⑨ in each of the 36 transformer layers. Qwen3-8B uses H=32 query heads and G=8 KV heads (group_size=4).

```
┌──────────────────────────────────────────────────────────────────────────┐
│  Inputs: q [1, 32, 128], k_cache [T, 8, 128], v_cache [T, 8, 128]      │
│                                                                          │
│  For each of 32 Q-heads h (in parallel):                                │
│    kv_h = h // 4   ← one of 8 KV-heads serves this Q-head               │
│                                                                          │
│    scores [T] = q[h] @ k_cache[:, kv_h, :].T / sqrt(128)              │
│    weights[T] = softmax(scores)   (no causal mask at decode)            │
│    out[h]     = weights @ v_cache[:, kv_h, :]  → [128]                 │
│                                                                          │
│  Reshape out: [32, 128] → [4096]                                        │
│  → O projection W[4096, 4096] → [1, 4096]                              │
└──────────────────────────────────────────────────────────────────────────┘
```

### KV cache size over a conversation (Qwen3-8B)

```
After T tokens:
  k_cache: [T, 8, 128] BF16 = T × 8 × 128 × 2 = 2,048 × T bytes ≈ 2T KB
  v_cache: same
  Both per layer: 4T KB

  × 36 layers: 144T KB total KV cache

  T =  1,000:   144 MB  (fits easily on RTX 4080)
  T =  4,096:   590 MB
  T = 32,768:  4.7 GB   (requires FP8 weights to leave room)
```

### GQA bandwidth at decode vs weight bandwidth

```
At T=4096 decode step:
  KV cache read: T × 8 × 128 × 2 × 2 bytes (K and V) = 16 MB
  Weight reads:  380 MB (all 7 projection matrices)

  KV cache = 4.2% of total HBM traffic
```

At short to medium context, the KV cache read is dominated by weight reads. Only at T > ~10,000 does the KV cache become a significant fraction of total memory traffic — which is exactly when GQA's 4× reduction matters most.

## GQA Implementations in the Wild

### FlashAttention-2 / FlashInfer

FlashInfer's `single_prefill_with_kv_cache` and `batch_decode_with_kv_cache` both support GQA natively.
The `num_qo_heads` and `num_kv_heads` parameters control the grouping:

```python
flashinfer.batch_decode_with_kv_cache(
    q, kv_cache, num_qo_heads=32, num_kv_heads=8
)
```

Internally: each thread block serving Q-head `h` loads KV from head `h // group_size`.

### vLLM

`vllm/attention/backends/flash_attn.py` — passes GQA config directly to FlashAttention-2.
`vllm/attention/backends/paged_attn.py` — custom PagedAttention kernel with native GQA:

```c
// In paged_attention_v2_kernel.cu:
const int kv_head_idx = head_idx / RATIO;   // RATIO = num_q_heads / num_kv_heads
```

### xFormers

`xformers.ops.memory_efficient_attention` — handles GQA by broadcasting KV heads before the kernel:
```python
# [B, T, G, D] → [B, T, H, D] via expand
k_expanded = k.repeat_interleave(group_size, dim=2)
```
Simpler than native GQA but uses more memory (stores H copies instead of G).

### HuggingFace Transformers

`modeling_utils.py::repeat_kv` — the standard expand-and-reshape approach:
```python
def repeat_kv(hidden_states, n_rep):
    # [B, T, G, D] → [B, T, G*n_rep, D]
    return hidden_states.unsqueeze(3).expand(..., n_rep, ...).reshape(...)
```
Appropriate for prefill or short-context inference. At long context, native GQA matters more.

### Native GQA vs expand-then-attention

| Approach | K/V memory | Code complexity | When to use |
|----------|-----------|-----------------|-------------|
| Expand K/V before attention | H×T×D bytes | Simple | Prefill, small T |
| Native GQA kernel | G×T×D bytes | Complex | Decode, large T, memory-constrained |

At T=32768: the K/V tensor is 4.7 GB at MHA vs 1.2 GB at GQA — a 4× reduction that is the difference between fitting on one RTX 4080 or needing two.

## One-Sentence Takeaway

GQA is the single architectural choice that makes long-context LLM serving practical on a consumer GPU: by sharing one KV head across four query heads, Qwen3-8B's KV cache is 4× smaller than standard MHA at every sequence length (enabling 4096-token conversations under 600 MB of KV memory and 32K-token sessions under 5 GB), and the implementation cost is literally one line of code (`kv_h = q_h // group_size`) that changes nothing about the attention math except which stored K and V vectors each query head reads from.

## Community Optimization Resources for Grouped Query Attention

### Measured speedups on SM89-class hardware

| Source | Impl | HW | Before | After | Speedup | Notes |
|---|---|---|---|---|---|---|
| [qwen3-tts-triton](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | **RTX 4090 (SM89)** | 1× | **5.16×** | Triton + CUDA graphs + static KV | Same SM89 arch — best directly applicable result |
| [qwen3-tts-triton](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | RTX 5090 | 1× | 4.0× | CUDA graphs + static KV cache alone | 4× with no kernel changes at all |
| [qwen3.5-triton](https://github.com/RightNow-AI/qwen3.5-triton) | Triton | B200 | 29.5 tok/s | 92.5 tok/s | **3.1×** | CUDA graphs + kernel fusion |
| [LMSYS Qwen latency](https://www.lmsys.org/blog/2026-02-11-Qwen-latency/) | CUDA | MI300X | — | — | 15–20% | KV-cache layout change alone; ratio transfers |

### KV cache bandwidth at decode: the real bottleneck

Qwen3-8B has 8 KV heads and `head_dim=128` in BF16:

```
Per-layer KV cache bandwidth per decode token:
  2 (K+V) × T_ctx × 8 heads × 128 dims × 2 bytes

  T_ctx =  1,000:   4 MB/layer × 36 layers =  144 MB  →  0.38 ms/token
  T_ctx =  4,096:  16 MB/layer × 36 layers =  576 MB  →  1.52 ms/token
  T_ctx = 32,768: 128 MB/layer × 36 layers =  4.6 GB  → 12.1  ms/token

GQA impact (Qwen3 uses 8 KV heads, not 32):
  MHA (32 KV heads) at T_ctx=4096: 2,304 MB → 6.1 ms/token
  GQA ( 8 KV heads) at T_ctx=4096:   576 MB → 1.52 ms/token  ← 4× smaller
```

**No kernel optimization reduces this.** The bytes must be read. The levers are:
GQA (already in Qwen3), FP8 KV (halves again to ~0.76 ms at T_ctx=4096), batch size
(amortizes weight reads across users), and CUDA graphs (eliminates launch overhead).

### Implementations to study

| Repo / file | Impl | What it shows |
|---|---|---|
| [qwen3-tts-triton](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | End-to-end RTX 4090 result; CUDA graphs vs kernel fusion breakdown |
| [qwen3.5-triton](https://github.com/RightNow-AI/qwen3.5-triton) | Triton | Profiler breakdown: 80.4% cuBLAS, 2.5% launch overhead |
| [LMSYS Qwen latency](https://www.lmsys.org/blog/2026-02-11-Qwen-latency/) | CUDA | KV cache layout that gives 15–20% improvement (reorders K/V interleaved for cache line efficiency) |

### Flash-Decoding / Split-KV for long contexts

At decode with 32 Q heads and long T_ctx, only 32 CTAs run — leaving 44 of 76 SMs
idle on the RTX 4080 Laptop. Flash-Decoding fixes this by splitting the context:

```
4 chunks × T_ctx/4 each → 32 heads × 4 chunks = 128 CTAs
SM utilization: 128 / 76 ≈ 1.7× — much better
```

Sources that implement or describe this:
- [LMSYS Qwen latency blog](https://www.lmsys.org/blog/2026-02-11-Qwen-latency/) — describes split-KV decode
- [qwen3.5-triton](https://github.com/RightNow-AI/qwen3.5-triton) — persistent kernel approach to amortize overhead

### CuTeDSL GQA kernel sketch

```python
@cute.kernel
def gqa_kernel(mQ, mK, mV, mOut, scale, group_size, T_ctx):
    h       = blockIdx.x           # query head index
    kv_h    = h // group_size      # → 0..7 for Qwen3-8B (group_size=4)
    m       = threadIdx.x          # query position (batch=1 at decode)

    rAcc = zeros_like_accum()
    for s in cute.range(T_ctx):
        score = dot(mQ[h, m, :], mK[kv_h, s, :]) * scale
        # ... online softmax update ...
        rAcc += weight * mV[kv_h, s, :]

    mOut[h, m, :] = rAcc / normalizer
```

The only GQA-specific line is `kv_h = h // group_size`. All kernel optimizations
(tiling, pipelining, async copy) are identical to standard MHA.